In [ ]:
def american_to_raw_implied(ml):
    """Raw implied prob from American odds; NaN if missing or non-numeric (e.g. 'NL')."""
    if ml is None:
        return np.nan
    try:
        if pd.isna(ml):
            return np.nan
    except TypeError:
        pass
    try:
        ml = float(ml)
    except (TypeError, ValueError):
        return np.nan
    if ml < 0:
        return abs(ml) / (abs(ml) + 100.0)
    return 100.0 / (ml + 100.0)


def american_to_decimal(ml):
    if ml is None:
        return np.nan
    try:
        if pd.isna(ml):
            return np.nan
    except TypeError:
        pass
    try:
        ml = float(ml)
    except (TypeError, ValueError):
        return np.nan
    if ml < 0:
        return 1.0 + 100.0 / abs(ml)
    return 1.0 + ml / 100.0


def devig_two_way_p_home(p_h_raw, p_a_raw):
    s = p_h_raw + p_a_raw
    out_h = p_h_raw / s
    out_a = p_a_raw / s
    bad = (s <= 0) | pd.isna(s)
    out_h = out_h.where(~bad, np.nan)
    out_a = out_a.where(~bad, np.nan)
    return out_h, out_a


# 2022–2024 park factors (100 = league average; feature = value/100). Full 30-team table from user source.
# master_mlb.csv uses ATH for Athletics (table listed Oakland Coliseum for this window).
PARK_FACTORS = {
    "ARI": 101,
    "ATL": 100,
    "ATH": 97,
    "BAL": 99,
    "BOS": 107,
    "CHC": 97,
    "CHW": 99,
    "CIN": 105,
    "CLE": 97,
    "COL": 112,
    "DET": 98,
    "HOU": 100,
    "KCR": 104,
    "LAA": 100,
    "LAD": 100,
    "MIA": 101,
    "MIL": 97,
    "MIN": 102,
    "NYM": 97,
    "NYY": 100,
    "PHI": 101,
    "PIT": 101,
    "SDP": 96,
    "SEA": 91,
    "SFG": 97,
    "STL": 100,
    "TBR": 96,
    "TEX": 101,
    "TOR": 100,
    "WSN": 101,
}


def add_market_columns(df):
    df = df.copy()
    oh_ml = pd.to_numeric(df["open_home_ml"], errors="coerce")
    oa_ml = pd.to_numeric(df["open_away_ml"], errors="coerce")
    ch_ml = pd.to_numeric(df["close_home_ml"], errors="coerce")
    ca_ml = pd.to_numeric(df["close_away_ml"], errors="coerce")

    oh = oh_ml.map(american_to_raw_implied)
    oa = oa_ml.map(american_to_raw_implied)
    ch = ch_ml.map(american_to_raw_implied)
    ca = ca_ml.map(american_to_raw_implied)

    # Rows where open is "NL" / missing: treat open like close so line_move / sharp flags are neutral
    oh = oh.where(oh.notna(), ch)
    oa = oa.where(oa.notna(), ca)

    df["open_home_implied"], _ = devig_two_way_p_home(oh, oa)
    df["market_implied_prob"], _ = devig_two_way_p_home(ch, ca)
    df["line_move_delta"] = df["market_implied_prob"] - df["open_home_implied"]
    df["sharp_move_flag"] = (df["line_move_delta"].abs() >= 0.03).astype(float)
    ot = pd.to_numeric(df["open_total"], errors="coerce")
    ct = pd.to_numeric(df["close_total"], errors="coerce")
    df["total_move_delta"] = (ct - ot).fillna(0.0)
    df["dec_close_home"] = ch_ml.map(american_to_decimal)
    return df


def add_schedule_context(df):
    df = df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])
    df = df.sort_values("game_date").reset_index(drop=True)
    df["park_factor"] = df["home_team"].map(lambda t: float(PARK_FACTORS.get(t, 100)) / 100.0)
    deg = np.deg2rad(df["wind_dir_deg"].astype(float).fillna(0))
    df["wind_dir_sin"] = np.sin(deg)
    df["wind_dir_cos"] = np.cos(deg)

    rows = []
    for _, r in df.iterrows():
        for side, team in [("home", r["home_team"]), ("away", r["away_team"])]:
            rows.append(
                {
                    "game_id": r["game_id"],
                    "game_date": r["game_date"],
                    "season": r["season"],
                    "team": team,
                    "side": side,
                }
            )
    g = pd.DataFrame(rows).sort_values(["team", "season", "game_date"])
    g["prev_date"] = g.groupby(["team", "season"], sort=False)["game_date"].shift(1)
    g["rest_days"] = (g["game_date"] - g["prev_date"]).dt.days.fillna(4).clip(lower=0)
    rh = g[g["side"] == "home"][["game_id", "rest_days"]].rename(columns={"rest_days": "home_rest"})
    ra = g[g["side"] == "away"][["game_id", "rest_days"]].rename(columns={"rest_days": "away_rest"})
    df = df.merge(rh, on="game_id", how="left").merge(ra, on="game_id", how="left")
    df["rest_days_DIFF"] = df["home_rest"] - df["away_rest"]
    # master_mlb.csv has no SP-specific rest; proxy = team off-days diff (see mlb_pipeline.md).
    df["sp_rest_DIFF"] = df["rest_days_DIFF"]

    tmp = df.sort_values(["home_team", "season", "game_date"]).copy()
    nxt = tmp.groupby(["home_team", "season"])["away_team"].shift(-1)
    tmp["is_series_finale"] = ((nxt != tmp["away_team"]) | nxt.isna()).astype(float)
    df = df.drop(columns=["is_series_finale"], errors="ignore").merge(
        tmp[["game_id", "is_series_finale"]], on="game_id", how="left"
    )
    return df


def load_master_csv(path=DATA_PATH):
    df = pd.read_csv(path, low_memory=False)
    df = add_market_columns(df)
    df = add_schedule_context(df)
    return df


df_raw = load_master_csv()
print(df_raw.shape)
df_raw[["game_date", "home_team", "market_implied_prob"]].head()

In [ ]:
DROP_SUBSTR = {"game_pk", "odds_source", "starter_id"}


def shift_team_perf(df):
    base_ids = ["game_id", "game_date", "season", "home_team", "away_team", "home_win", "home_score", "away_score"]
    home_cols = [
        c
        for c in df.columns
        if c.startswith("home_")
        and c
        not in {
            "home_team",
            "home_win",
            "home_score",
            "home_starter_id",
            "home_pitcher_is_lefty",
        }
        and not any(s in c for s in DROP_SUBSTR)
    ]
    away_cols = [
        c
        for c in df.columns
        if c.startswith("away_")
        and c
        not in {
            "away_team",
            "away_score",
            "away_starter_id",
            "away_pitcher_is_lefty",
        }
        and not any(s in c for s in DROP_SUBSTR)
    ]

    h = df[base_ids + ["home_pitcher_is_lefty"] + home_cols].copy()
    h = h.rename(columns={c: c.replace("home_", "", 1) for c in home_cols})
    h["team"] = df["home_team"]
    h["opp"] = df["away_team"]
    h["is_home"] = 1.0
    h["win"] = df["home_win"].astype(float)
    h["runs_for"] = df["home_score"].astype(float)
    h["runs_against"] = df["away_score"].astype(float)

    a = df[base_ids + ["away_pitcher_is_lefty"] + away_cols].copy()
    a = a.rename(columns={c: c.replace("away_", "", 1) for c in away_cols})
    a["team"] = df["away_team"]
    a["opp"] = df["home_team"]
    a["is_home"] = 0.0
    a["win"] = 1.0 - df["home_win"].astype(float)
    a["runs_for"] = df["away_score"].astype(float)
    a["runs_against"] = df["home_score"].astype(float)

    long = pd.concat([h, a], ignore_index=True)
    long = long.sort_values(["team", "season", "game_date", "game_id"])
    stat_cols = [
        c
        for c in long.columns
        if c
        not in base_ids + ["team", "opp", "is_home", "win", "runs_for", "runs_against"]
    ]
    for c in stat_cols:
        long[c] = pd.to_numeric(long[c], errors="coerce")
    for c in stat_cols:
        long[c + "_lag1"] = long.groupby(["team", "season"], sort=False)[c].shift(1)

    lag_cols = [c for c in long.columns if c.endswith("_lag1")]
    long = long.drop(columns=stat_cols, errors="ignore")

    hm = long[long["is_home"] == 1][["game_id"] + lag_cols].copy()
    hm.columns = ["game_id"] + ["h_" + c for c in lag_cols]
    aw = long[long["is_home"] == 0][["game_id"] + lag_cols].copy()
    aw.columns = ["game_id"] + ["a_" + c for c in lag_cols]

    out = df.merge(hm, on="game_id", how="left").merge(aw, on="game_id", how="left")
    return out, long


def add_rolling_form(long_games, W):
    g = long_games.sort_values(["team", "season", "game_date", "game_id"]).copy()

    def rmean(ser, window):
        return ser.shift(1).rolling(window, min_periods=max(2, window // 3)).mean()

    g[f"win_pct_{W}"] = g.groupby(["team", "season"], sort=False)["win"].transform(lambda s: rmean(s, W))
    g["run_margin"] = g["runs_for"] - g["runs_against"]
    g[f"run_diff_avg_{W}"] = g.groupby(["team", "season"], sort=False)["run_margin"].transform(
        lambda s: rmean(s, W)
    )
    g[f"runs_scored_avg_{W}"] = g.groupby(["team", "season"], sort=False)["runs_for"].transform(
        lambda s: rmean(s, W)
    )
    roll_cols = [f"win_pct_{W}", f"run_diff_avg_{W}", f"runs_scored_avg_{W}"]
    hm = g[g["is_home"] == 1][["game_id"] + roll_cols].copy()
    hm.columns = ["game_id"] + [f"h_{c}" for c in roll_cols]
    aw = g[g["is_home"] == 0][["game_id"] + roll_cols].copy()
    aw.columns = ["game_id"] + [f"a_{c}" for c in roll_cols]
    return hm.merge(aw, on="game_id")


df_shift, long_games = shift_team_perf(df_raw)


def eval_brier_for_W(df_base, W, lg):
    roll = add_rolling_form(lg, W)
    d = df_base.merge(roll, on="game_id", how="inner")
    d = d[d["season"] <= TRAIN_SEASON_MAX]
    feats = [
        "market_implied_prob",
        f"h_win_pct_{W}",
        f"a_win_pct_{W}",
        f"h_run_diff_avg_{W}",
        f"a_run_diff_avg_{W}",
        "h_fip_lag1",
        "a_fip_lag1",
    ]
    feats = [c for c in feats if c in d.columns]
    dd = d.dropna(subset=feats + ["home_win"])
    if len(dd) < 5000:
        return np.nan
    years = sorted(dd["season"].unique())
    scores = []
    for y in years[-5:]:
        tr = dd[dd["season"] < y]
        va = dd[dd["season"] == y]
        if len(tr) < 1000 or len(va) < 100:
            continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(tr[feats].values)
        Xva = sc.transform(va[feats].values)
        m = LogisticRegression(max_iter=300)
        m.fit(Xtr, tr["home_win"])
        p = m.predict_proba(Xva)[:, 1]
        scores.append(brier_score_loss(va["home_win"], p))
    return float(np.mean(scores)) if scores else np.nan


brier_by_w = {W: eval_brier_for_W(df_shift, W, long_games) for W in ROLL_WINDOWS}
BEST_W = int(min(brier_by_w, key=lambda w: (np.nan_to_num(brier_by_w[w], nan=1e9))))
print("Brier by W:", brier_by_w, "-> BEST_W =", BEST_W)

roll_merge = add_rolling_form(long_games, BEST_W)
df_feat = df_shift.merge(roll_merge, on="game_id", how="left")

# DIFF signs per mlb_pipeline.md (season-to-date / lagged team-starter fields).
df_feat["sp_fip_DIFF"] = df_feat["a_fip_lag1"] - df_feat["h_fip_lag1"]
df_feat["sp_k9_DIFF"] = df_feat["h_sp_k9_lag1"] - df_feat["a_sp_k9_lag1"]
df_feat["rolling_k9_DIFF"] = df_feat["h_rolling_k9_lag1"] - df_feat["a_rolling_k9_lag1"]
df_feat["wrc_plus_DIFF"] = df_feat["h_wrc_plus_lag1"] - df_feat["a_wrc_plus_lag1"]
df_feat["woba_DIFF"] = df_feat["h_woba_lag1"] - df_feat["a_woba_lag1"]
df_feat["war_DIFF"] = df_feat["h_war_lag1"] - df_feat["a_war_lag1"]
df_feat["pitcher_handedness_diff"] = df_feat["home_pitcher_is_lefty"] - df_feat["away_pitcher_is_lefty"]
df_feat["win_pct_W_DIFF"] = df_feat[f"h_win_pct_{BEST_W}"] - df_feat[f"a_win_pct_{BEST_W}"]
df_feat["run_diff_avg_W_DIFF"] = df_feat[f"h_run_diff_avg_{BEST_W}"] - df_feat[f"a_run_diff_avg_{BEST_W}"]
df_feat["runs_scored_avg_W_DIFF"] = df_feat[f"h_runs_scored_avg_{BEST_W}"] - df_feat[f"a_runs_scored_avg_{BEST_W}"]

df_feat = df_feat.sort_values("game_date").reset_index(drop=True)
df_feat["hg"] = df_feat.groupby(["home_team", "season"]).cumcount()
df_feat["ag"] = df_feat.groupby(["away_team", "season"]).cumcount()
df_feat["early_season_flag"] = (df_feat[["hg", "ag"]].min(axis=1) < EARLY_SEASON_GAMES).astype(float)

FEATURE_COLUMNS = [
    "market_implied_prob",
    "open_home_implied",
    "line_move_delta",
    "sharp_move_flag",
    "total_move_delta",
    "sp_fip_DIFF",
    "sp_k9_DIFF",
    "rolling_k9_DIFF",
    "wrc_plus_DIFF",
    "woba_DIFF",
    "war_DIFF",
    "pitcher_handedness_diff",
    "home_pitcher_is_lefty",
    "away_pitcher_is_lefty",
    "win_pct_W_DIFF",
    "run_diff_avg_W_DIFF",
    "runs_scored_avg_W_DIFF",
    "temp_c",
    "wind_dir_sin",
    "wind_dir_cos",
    "park_factor",
    "is_night_game",
    "rest_days_DIFF",
    "sp_rest_DIFF",
    "is_series_finale",
]

FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c in df_feat.columns]
print("n features:", len(FEATURE_COLUMNS), FEATURE_COLUMNS)
df_feat[FEATURE_COLUMNS + ["home_win", "season", "game_date"]].dropna().shape